# 04 — Validação temporal e seleção (seções 13–16 do v5)

**Lê:** `painel` (só linhas de treino), `panel_base` até `TRAIN_TARGET_END`, `painel_spec.json`.  
**Grava:** `cv_detalhe`, `resumo_validacao`, **`selecao_modelo.json`** (com o hash do painel).

Nenhuma linha de teste é carregada. A seleção é congelada aqui e destrava o notebook 05.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "04_validacao_selecao"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import cv, modelos, Especificacao

spec = Especificacao.de_json(A.carregar_json("painel_spec"), C)
train = A.carregar_treino()
hist = A.carregar_historico_treino()
BOOST, RIDGE_NAMES, CV_MODELS = modelos.lista_modelos(C.RIDGE_ALPHAS)
cv_origins, dropped = cv.origens_cv(C)
print("Modelos na validação:", CV_MODELS)
print(f"Origens completas ({len(cv_origins)}): {[o.date() for o in cv_origins]} | removidas: {dropped or 'nenhuma'}")

## 14. Validação temporal (somente dentro do treino)

In [ ]:
cv_detail, ridge_q = cv.rodar_cv(train, hist, cv_origins, CV_MODELS, spec, C)
A.salvar_df(cv_detail, "cv_detalhe")
A.tabela(cv_detail, "validacao_temporal_detalhada.csv")

## 15. Estabilidade do Ridge
Instável se, em alguma origem, gerar previsão não finita ou acima do maior alvo do treino da própria origem.

In [ ]:
display(ridge_q)
ridge_stab, RIDGE_NAME, RIDGE_ELIGIBLE = cv.estabilidade_ridge(cv_detail, RIDGE_NAMES)
display(ridge_stab)
print(f"Ridge {'estável: ' + RIDGE_NAME if RIDGE_ELIGIBLE else 'REMOVIDO da seleção (documentado: ' + RIDGE_NAME + ')'}")

## 16. Resumo da validação e seleção congelada

In [ ]:
resumo = cv.resumo_validacao(cv_detail, BOOST, RIDGE_NAME, RIDGE_ELIGIBLE)
A.salvar_df(resumo, "resumo_validacao")
A.tabela(resumo, "resumo_validacao.csv")
display(resumo.sort_values("MASE_vend_media"))

selection = cv.selecionar(resumo, cv_origins, C, RIDGE_NAME, RIDGE_ELIGIBLE, painel_sha256=A.sha("painel"))
selection["boost_models"] = BOOST
A.salvar_json(selection, "selecao_modelo")
print({k: selection[k] for k in ["modelo_escolhido", "criterio", "valor_criterio", "modelos_em_empate_pratico"]})

In [ ]:
A.finalizar()